In [ ]:
import time
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, fpgrowth

# Doc du lieu: moi dong cua file la 1 giao dich, cac muc tach nhau bang dau phay
transactions = []
with open("groceries.csv", "r", encoding="utf-8") as f:
    for line in f:
        items = [item.strip() for item in line.strip().split(",") if item.strip()]
        if items:
            transactions.append(items)

# Chuyen doi sang one-hot encoding
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
basket = pd.DataFrame(te_ary, columns=te.columns_).astype(bool)
print("Bang one-hot:", basket.shape)

ds_min_sup = [0.0001, 0.0002, 0.0003, 0.0004, 0.0005,
              0.0006, 0.0007, 0.0008, 0.0009, 0.0010]

# Cau 6a: tap pho bien bang FP-Growth, ghi nhan thoi gian
ket_qua = []
for min_sup in ds_min_sup:
    start = time.time()
    itemsets = fpgrowth(basket, min_support=min_sup, use_colnames=True, max_len=2)
    elapsed = time.time() - start
    ket_qua.append({
        "min_sup (%)": round(min_sup * 100, 2),
        "so_tap_pho_bien": len(itemsets),
        "FP-Growth (s)": round(elapsed, 4)
    })
    print(f"min_sup = {min_sup*100:.2f}%: {len(itemsets)} tap pho bien | {elapsed:.4f}s")

bang_fp = pd.DataFrame(ket_qua)
print("\n--- BANG TONG HOP KET QUA FP-GROWTH ---")
display(bang_fp)

# Cau 6b: do thoi gian Apriori tren cung cac nguong de so sanh
thoi_gian_apriori = []
for min_sup in ds_min_sup:
    start = time.time()
    ap = apriori(basket, min_support=min_sup, use_colnames=True, max_len=2)
    thoi_gian_apriori.append(round(time.time() - start, 4))

bang_fp["Apriori (s)"] = thoi_gian_apriori
print("\n--- BANG SO SANH THOI GIAN APRIORI vs FP-GROWTH ---")
display(bang_fp)

# Cau 6c: ve bieu do so sanh thoi gian xu ly
plt.figure(figsize=(12, 6))
x = range(len(ds_min_sup))
plt.plot(x, bang_fp["Apriori (s)"], marker="o", linewidth=2,
         color="tab:red", label="Apriori")
plt.plot(x, bang_fp["FP-Growth (s)"], marker="s", linewidth=2,
         color="tab:blue", label="FP-Growth")
plt.xticks(list(x), [f"{s*100:.2f}%" for s in ds_min_sup], rotation=45)
plt.xlabel("min_sup")
plt.ylabel("Thời gian xử lý (giây)")
plt.title("So sánh thời gian xử lý giữa Apriori và FP-Growth (Groceries)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Xem tap pho bien chi tiet voi min_sup = 0.1%
itemsets_fp = fpgrowth(basket, min_support=0.001, use_colnames=True, max_len=2)
itemsets_fp["do_dai"] = itemsets_fp["itemsets"].apply(len)
print("\n--- 10 tap pho bien co support cao nhat (FP-Growth, min_sup = 0.1%) ---")
top = itemsets_fp.sort_values("support", ascending=False).head(10).copy()
top["support"] = top["support"].round(4)
top["itemsets"] = top["itemsets"].apply(lambda s: ", ".join(sorted(s)))
display(top[["support", "itemsets"]])